# Inference, evaluation, and comparison

Last updated 2026-11-24 00:40 JST

Use the training `EXPERIMENT_ID`. Restore the completed Baseline and selected transfer checkpoints from verified ZIPs, then evaluate with the shared generator and dev split. By default, evaluate the Baseline. Enable the matching `COMPARE` switches for completed transfer runs. [training.ipynb](training.ipynb)

Use `DATA_CONFIG` or the input overrides to select the same corpus root and manifest used in training. Use the original experiment ID for saved models.

## GitHub access

For a private repository, create a fine-grained personal access token in GitHub **Settings → Developer settings → Personal access tokens → Fine-grained tokens**. Select the repository owner, choose **Only select repositories**, select `alignment-transfer-s2ut`, and grant **Contents → Read-only**. Set an expiration date. See the [GitHub token guide](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens).

In Colab, open **Secrets** (the key icon), add `GITHUB_TOKEN`, paste the token into its value, and enable **Notebook access**. The setup cell reads this secret for Git operations. Public repositories also work without a token. Successful clone/fetch and checkout in the setup cell confirm access to the requested revision.

To authenticate an additional repository, add its exact HTTPS URL to `GITHUB_AUTH_REPOS` in the settings cell and grant the same token read access to that repository. The benchmark is public and uses anonymous HTTPS by default.

After a runtime reset, use the same notebook revision, experiment settings, and Drive paths, then rerun the cells. To open the notebook after the repository becomes private, download the `.ipynb` while signed into GitHub and use **File → Upload notebook** in Colab, or open a copy saved in your Drive. Enable secret access for that notebook. The runtime token is used after the notebook opens.


In [ ]:
#@title Process logging and progress
from pathlib import Path
import json
"""Standard-library-only process logging, also embedded in Colab before install."""


def log_event(name, state, logs, detail=""):
    from datetime import datetime, timezone
    from pathlib import Path
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    line = f"{datetime.now(timezone.utc).isoformat()} [{name}] {state} {detail}\n"
    with (root / "notebook-events.log").open("a", encoding="utf-8") as stream:
        stream.write(line)
        stream.flush()
    print(line, end="", flush=True)


def run_process(command, logs, *, name="command", capture_json=False, env=None, cwd=None,
                heartbeat_seconds=10., extra_record=None):
    """Stream every child line; silence triggers liveness, never fake progress.

    stdout JSON is kept apart from stderr. Nested loggers share an owned POSIX
    process group so interruption cannot strand a second training process.
    """
    from datetime import datetime, timezone
    import hashlib
    import json
    import os
    from pathlib import Path
    import queue
    import signal
    import subprocess
    import tempfile
    import threading
    import time

    if heartbeat_seconds <= 0:
        raise ValueError("heartbeat_seconds must be positive")
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    command = list(map(str, command))
    key = hashlib.sha256(json.dumps(name, ensure_ascii=False).encode()).hexdigest()[:12] + "-" + str(time.time_ns())
    output, result_path, receipt = root / (key + ".log"), root / (key + ".stdout.log"), root / (key + ".json")
    launch_env = dict(os.environ if env is None else env)
    launch_env["PYTHONUNBUFFERED"] = "1"
    owns_group = os.name == "posix" and launch_env.get("AT_LOG_MANAGED_GROUP") != "1"
    if owns_group:
        launch_env["AT_LOG_MANAGED_GROUP"] = "1"
    started, clock = datetime.now(timezone.utc).isoformat(), time.perf_counter()
    record = dict(name=name, command=command, started_utc=started,
        host_utc_offset_seconds=int(datetime.now().astimezone().utcoffset().total_seconds()),
        elapsed_seconds=0., returncode=None, status="running", exception=None,
        pid=None, log=str(output.resolve()), stdout_log=str(result_path.resolve()) if capture_json else None,
        gpu_csv=None, gpu_monitor_error=None, cwd=str(Path(cwd or Path.cwd()).resolve()))
    lock, stop, done = threading.Lock(), threading.Event(), threading.Event()
    messages, reader_errors = queue.Queue(maxsize=256), []
    process = reader = result_stream = None
    stream = output.open("w", encoding="utf-8")

    def save_record():
        record["elapsed_seconds"] = time.perf_counter() - clock
        fd, temporary = tempfile.mkstemp(dir=root, prefix=".log-record-")
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as target:
                json.dump(record, target, ensure_ascii=False, indent=2, allow_nan=False)
                target.write("\n")
                target.flush()
            os.replace(temporary, receipt)
        finally:
            if os.path.exists(temporary):
                os.unlink(temporary)

    def notice(state, detail=""):
        line = f"[{name}] {state} pid={record['pid']} elapsed={time.perf_counter()-clock:.1f}s {detail}\n"
        with lock:
            stream.write(line)
            stream.flush()
        print(line, end="", flush=True)

    def read_lines(pipe):
        try:
            for line in pipe:
                with lock:
                    stream.write(line)
                    stream.flush()
                while not stop.is_set():
                    try:
                        messages.put(line, timeout=.1)
                        break
                    except queue.Full:
                        pass
        except Exception as error:
            reader_errors.append(error)
        finally:
            done.set()

    def terminate_child():
        if process is None:
            return
        if owns_group:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
        elif process.poll() is None:
            process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            if owns_group:
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            process.wait()
        if owns_group:
            # A nested child may outlive its exited leader after cancellation.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass

    try:
        notice("started", f"full log: {output.resolve()}")
        save_record()
        if capture_json:
            result_stream = result_path.open("w", encoding="utf-8")
        process = subprocess.Popen(command, stdout=result_stream if capture_json else subprocess.PIPE,
            stderr=subprocess.PIPE if capture_json else subprocess.STDOUT, text=True,
            encoding="utf-8", errors="replace", bufsize=1, env=launch_env, cwd=cwd,
            start_new_session=owns_group)
        record["pid"] = process.pid
        notice("running", "waiting for child output")
        save_record()
        pipe = process.stderr if capture_json else process.stdout
        reader = threading.Thread(target=read_lines, args=(pipe,), daemon=True, name="command-log-reader")
        reader.start()
        last_output = last_notice = time.perf_counter()
        while not (done.is_set() and messages.empty() and process.poll() is not None):
            try:
                line = messages.get(timeout=min(.1, heartbeat_seconds))
            except queue.Empty:
                line = None
            if line is not None:
                print(line, end="", flush=True)
                if line.strip():
                    last_output = time.perf_counter()
            if reader_errors:
                raise reader_errors[0]
            now = time.perf_counter()
            if now - max(last_output, last_notice) >= heartbeat_seconds:
                notice("running", f"no_output={now-last_output:.1f}s (liveness only; progress not confirmed)")
                save_record()
                last_notice = now
        code = process.wait()
        if reader_errors:
            raise reader_errors[0]
        if capture_json and code == 0:
            json.loads(result_path.read_text(encoding="utf-8"))
        record.update(returncode=code, status="succeeded" if code == 0 else "failed_or_interrupted")
        if code:
            raise subprocess.CalledProcessError(code, command)
        notice("completed", f"exit={code}; full log: {output.resolve()}")
    except BaseException as error:
        record.update(status="failed_or_interrupted", exception=type(error).__name__ + ": " + str(error))
        terminate_child()
        if process is not None:
            record["returncode"] = process.returncode
        notice("interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
               f"exit={record['returncode']} {type(error).__name__}; full log: {output.resolve()}")
        raise
    finally:
        stop.set()
        if reader is not None:
            reader.join(timeout=2)
        if process is not None and (reader is None or not reader.is_alive()):
            pipe = process.stderr if capture_json else process.stdout
            if pipe is not None:
                pipe.close()
        if result_stream is not None:
            result_stream.close()
        if extra_record is not None:
            record.update(extra_record() if callable(extra_record) else extra_record)
        save_record()
        stream.close()
    result = subprocess.CompletedProcess(command, record["returncode"],
        result_path.read_text(encoding="utf-8") if capture_json else None)
    result.record = record
    return result


"""Standard-library GitHub authentication, embedded before Colab installation."""


GITHUB_GIT_WORKER = r'''
import base64
import os
import subprocess
import sys
from urllib.parse import quote

redactions = []
for i in range(int(os.environ.get("GIT_CONFIG_COUNT", "0"))):
    value = os.environ.get(f"GIT_CONFIG_VALUE_{i}", "")
    if value.startswith("Authorization: Basic "):
        encoded = value.split()[-1]
        token = base64.b64decode(encoded).decode("utf-8").split(":", 1)[1]
        redactions.extend((value, encoded, token, quote(token, safe="")))
redactions = sorted(set(redactions), key=len, reverse=True)
process = None
try:
    command = sys.argv[1:]
    position = 1
    while position < len(command) and command[position] == "-C":
        position += 2
    if command[position] in {"fetch", "pull"}:
        remote = next((arg for arg in command[position + 1:] if not arg.startswith("-")), "origin")
        if "://" not in remote:
            checked = subprocess.run([*command[:position], "remote", "get-url", "--all", remote],
                stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                text=True, encoding="utf-8", errors="replace")
            from urllib.parse import urlsplit
            for address in checked.stdout.splitlines():
                url = urlsplit(address)
                if url.username or url.password or url.query or url.fragment:
                    raise ValueError("unsafe stored remote")
    process = subprocess.Popen(command, stdin=subprocess.DEVNULL,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        encoding="utf-8", errors="replace", bufsize=1)
    for line in process.stdout:
        for secret in redactions:
            line = line.replace(secret, "[REDACTED]")
        print(line, end="", flush=True)
    code = process.wait()
except Exception:
    print("Git could not complete the requested operation.", file=sys.stderr, flush=True)
    code = 1
finally:
    if process is not None:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        process.stdout.close()
if code:
    print("Check the repository URL and revision. For a private repository, add "
          "GITHUB_TOKEN to Colab Secrets, enable notebook access, and grant the "
          "fine-grained token Contents: Read-only for the selected repository. "
          "Check token expiry and any organization approval.", file=sys.stderr, flush=True)
sys.exit(code)
'''


def run_github_git(command, logs, *, runner, repositories, name="git", env=None, cwd=None,
                   capture_json=False):
    """Authenticate network Git calls with URL-scoped, process-only headers."""
    import base64
    import os
    from pathlib import Path
    import re
    import sys
    from urllib.parse import urlsplit

    command = list(map(str, command))
    position = 1
    while position < len(command) and command[position] == "-C":
        position += 2
    operation = command[position] if position < len(command) else ""
    if operation not in {"clone", "fetch", "pull", "ls-remote"}:
        return runner(command, logs, name=name, env=env, cwd=cwd, capture_json=capture_json)
    if Path(command[0]).name.lower() not in {"git", "git.exe"}:
        raise ValueError("GitHub authentication requires a git command")
    for argument in command:
        if "://" in argument:
            url = urlsplit(argument)
            if url.username or url.password or url.query or url.fragment:
                raise ValueError("Use repository URLs without credentials, queries, or fragments")
    scopes = []
    for repository in repositories:
        url = urlsplit(str(repository))
        if (url.scheme != "https" or url.netloc != "github.com" or url.query or url.fragment
                or not re.fullmatch(r"/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", url.path)):
            raise ValueError("Authentication scopes must be plain HTTPS GitHub repository URLs")
        scopes.append(url.geturl())

    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass  # Public repositories remain accessible when Secrets are unavailable.
    if token is not None and (not isinstance(token, str) or any(ord(c) < 33 or ord(c) > 126 for c in token)):
        token = None
        raise ValueError("GITHUB_TOKEN must contain a valid GitHub token without whitespace")
    git_env = dict(os.environ if env is None else env)
    for key in list(git_env):
        if (key.startswith(("GIT_TRACE", "GIT_CONFIG_KEY_", "GIT_CONFIG_VALUE_"))
                or key in {"GIT_CONFIG_COUNT", "GIT_CONFIG_PARAMETERS", "GIT_CURL_VERBOSE", "GITHUB_TOKEN", "GH_TOKEN"}):
            git_env.pop(key)
    git_env.update(GIT_TERMINAL_PROMPT="0", GCM_INTERACTIVE="Never", GIT_ASKPASS="", GIT_TRACE_REDACT="1")
    settings = [("credential.helper", ""), ("http.extraHeader", ""), ("http.followRedirects", "false")]
    header = None
    try:
        if token:
            header = "Authorization: Basic " + base64.b64encode(("x-access-token:" + token).encode()).decode()
            settings.extend(("http." + scope + ".extraHeader", header) for scope in dict.fromkeys(scopes))
        git_env["GIT_CONFIG_COUNT"] = str(len(settings))
        for i, (key, value) in enumerate(settings):
            git_env[f"GIT_CONFIG_KEY_{i}"] = key
            git_env[f"GIT_CONFIG_VALUE_{i}"] = value
        return runner([sys.executable, "-c", GITHUB_GIT_WORKER, *command], logs,
                      name=name, env=git_env, cwd=cwd, capture_json=capture_json)
    except Exception:
        raise RuntimeError("GitHub access failed. Check the logged URL/revision error and Colab Secrets "
                           "GITHUB_TOKEN access, selected repositories, Contents: Read-only, and expiry.") from None
    finally:
        token = header = None
        settings.clear()
        git_env.clear()


def execute(*arguments, name=None, capture_json=False, env=None, cwd=None, bootstrap=False):
    release = globals().get('RELEASE_RECORD')
    if globals().get('RELEASE_REQUIRED') and release is None and not bootstrap:
        raise RuntimeError('Run the Colab revision check before starting the experiment.')

    def runner(command, logs, **options):
        metadata = None if release is None else {'colab_release': release}
        return run_process(command, logs, extra_record=metadata, **options)

    command = list(map(str, arguments))
    if name is None:
        name = command[2].rsplit('.', 1)[-1] if len(command) > 2 and command[1] == '-m' else Path(command[0]).name
        if '--method' in command:
            name += ':' + command[command.index('--method') + 1]
        if '--prior-type' in command:
            name += ':' + command[command.index('--prior-type') + 1]
    if command and command[0] == 'git':
        repositories = [globals().get('REPO_URL', 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'),
                        *globals().get('GITHUB_AUTH_REPOS', [])]
        return run_github_git(command, LOGS, runner=runner, repositories=repositories,
                              name=name, capture_json=capture_json, env=env, cwd=cwd)
    return runner(command, LOGS, name=name, capture_json=capture_json, env=env, cwd=cwd)


def execute_json(*arguments, name=None, env=None, cwd=None):
    return json.loads(execute(*arguments, name=name, capture_json=True, env=env, cwd=cwd).stdout)

LOGS = Path('/content/alignment-transfer-logs/evaluate/bootstrap')

In [ ]:
#@title Mount Google Drive

from google.colab import drive
log_event('drive-mount', 'started', LOGS)
try:
    drive.mount('/content/drive')
except BaseException as error:
    log_event('drive-mount', 'failed', LOGS, type(error).__name__)
    raise
else:
    log_event('drive-mount', 'completed', LOGS)
import json, os, subprocess, sys, time
from pathlib import Path


In [ ]:
#@title Experiment and conditions to evaluate {display-mode:"form"}

#@markdown ### Experiment
EXPERIMENT_ID = 'experiment-001' #@param {type:"string"}
TRANSFER_EXPERIMENT_ID = '' #@param {type:"string"}

#@markdown ### Corpus inputs
DATA_CONFIG = 'configs/data/colab.yaml' #@param {type:"string"}
CORPUS_ROOT_OVERRIDE = '' #@param {type:"string"}
SOURCE_MANIFEST_OVERRIDE = '' #@param {type:"string"}
RELEASE_SUMMARY_OVERRIDE = '' #@param {type:"string"}

#@markdown ### S2UT conditions and metrics
EVALUATE_S2UT = True #@param {type:"boolean"}
COMPARE_DISTILLATION = False #@param {type:"boolean"}
COMPARE_ANNEALED_DISTILLATION = False #@param {type:"boolean"}
COMPARE_MASK_DISTILLATION = False #@param {type:"boolean"}
GENERATE_WAVEFORMS = False #@param {type:"boolean"}
ASR_BLEU = False #@param {type:"boolean"}

#@markdown ### Transfer BLEU backfill
BACKFILL_TRANSFER_BLEU = False #@param {type:"boolean"}
TRANSFER_BLEU_UPDATE = 0 #@param {type:"integer"}
SYNC_SAVED_TRANSFER_BLEU = False #@param {type:"boolean"}
ENABLE_WANDB = False #@param {type:"boolean"}
WANDB_PROJECT = 'alignment-transfer-s2ut' #@param {type:"string"}
WANDB_ENTITY = '' #@param {type:"string"}

#@markdown ### T2TT external evaluation (optional)
EVALUATE_T2TT_EXTERNAL = False #@param {type:"boolean"}
T2TT_RUN_ID = 't2tt-001' #@param {type:"string"}
T2TT_MAX_UPDATES = 200000 #@param {type:"integer"}
T2TT_TOKENIZER = 'sentencepiece' #@param ['character', 'sentencepiece']
REFERENCE_BENCHMARK = '' #@param {type:"string"}
SOURCE_8C = '' #@param {type:"string"}

# Validate settings before setup.
if BACKFILL_TRANSFER_BLEU and (not EVALUATE_S2UT or not any(
        (COMPARE_DISTILLATION, COMPARE_ANNEALED_DISTILLATION, COMPARE_MASK_DISTILLATION))):
    raise ValueError('Enable S2UT and select at least one transfer condition for BLEU backfill.')
if BACKFILL_TRANSFER_BLEU and TRANSFER_BLEU_UPDATE < 0:
    raise ValueError('Set TRANSFER_BLEU_UPDATE to zero or an exact saved milestone.')
if SYNC_SAVED_TRANSFER_BLEU and (BACKFILL_TRANSFER_BLEU or EVALUATE_S2UT or EVALUATE_T2TT_EXTERNAL
        or not ENABLE_WANDB or not any((COMPARE_DISTILLATION, COMPARE_ANNEALED_DISTILLATION, COMPARE_MASK_DISTILLATION))):
    raise ValueError('For sync-only, select transfer conditions and enable W&B. Turn off speech evaluation, text evaluation and backfill.')
if TRANSFER_EXPERIMENT_ID:
    import re
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]*', TRANSFER_EXPERIMENT_ID):
        raise ValueError('TRANSFER_EXPERIMENT_ID must use letters, digits, underscores, or hyphens')
if ASR_BLEU and not GENERATE_WAVEFORMS:
    raise ValueError('Set GENERATE_WAVEFORMS=True to compute ASR-BLEU.')
PREPARE_AUDIO_PACKS = False


In [ ]:
#@title Storage and pinned environment
import os
import sys
for value in (EXPERIMENT_ID, globals().get('T2TT_RUN_ID', 'text')):
    if not value or Path(value).name != value or value in ('.', '..'):
        raise ValueError('Use a single directory name for the experiment ID.')
REPO_URL = 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'
NOTEBOOK_UPDATED_AT = '2026-11-24T00:40+09:00'
REPO_REVISION = 'f347a03f5b9b74037d7d8f1baef29827d6cf4dae'
REPO = Path('/content/alignment-transfer-s2ut')
PYTHON = '/content/alignment-venv/bin/python'
ROOT = str(Path('/content/alignment-transfer-work') / EXPERIMENT_ID)
PERSISTENT = str(Path('/content/drive/MyDrive/alignment-transfer-s2ut-data') / EXPERIMENT_ID)
LOGS = Path(ROOT) / 'execution'
BENCHMARK_RUNTIME_URL = 'https://github.com/Yaaamashiro/ja-en-direct-s2st-benchmark.git'
BENCHMARK_RUNTIME = Path('/content/alignment-benchmark-runtime')
BENCHMARK_RUNTIME_REVISION = '0c0024ea888206149a3119bf096b86ba66d63b66'
DATA = Path(ROOT) / 'data'
COMMON = DATA / 'common'
BENCHMARK_UNITS = DATA / 's2ut/units'
PREPARED = DATA / 's2ut/fairseq'
BACKUP = Path(PERSISTENT) / 'baseline'
RESTORED = Path(ROOT) / 'baseline'
BASELINE = Path(ROOT) / 'registrations/baseline.json'
MANIFEST = Path(ROOT) / 'manifest/manifest.jsonl'
BACKBONE_CONFIG = Path(ROOT) / 'backbone.json'
TRANSFER_SUFFIX = '-run-' + TRANSFER_EXPERIMENT_ID if TRANSFER_EXPERIMENT_ID else ''
SCOPE_SUFFIX = '-spm-all-layers' + TRANSFER_SUFFIX
RESEARCH_ROOT = Path(ROOT) / ('research-v3' + SCOPE_SUFFIX)
METHOD_LABELS = [
    ('distillation', 'KL-based Attention Distillation'),
    ('annealed_distillation', 'Annealed KL-based Attention Distillation'),
    ('mask_distillation', 'Attention-level Knowledge Distillation')]
os.environ.update(HF_HOME='/content/huggingface', MPLBACKEND='Agg', PYTHONUNBUFFERED='1', PYTHONUTF8='1',
    MPLCONFIGDIR='/content/matplotlib-cache', S2ST_DRIVE_SAFE='1',
    S2ST_BACKUP_MIN_SECONDS='600', S2ST_BACKUP_MAX_GB='1024')
RELEASE_REQUIRED = True
RELEASE_RECORD = None


## 1. Set up and check the environment

In [ ]:
#@title Verify Git revisions and record the notebook update time
"""Validate the two Colab checkouts and synchronize the embedded bootstrap."""

from datetime import datetime, timezone
import json
from pathlib import Path
import re
import sys
import uuid


GIT_CHECKOUT_STATE = r"""
import json
import subprocess
import sys

def read_git(path, *arguments):
    result = subprocess.run(
        ["git", "-C", path, *arguments], stdout=subprocess.PIPE, check=False,
        encoding="utf-8", errors="replace", text=True)
    return result.stdout if result.returncode == 0 else None

states = []
for path in sys.argv[1:]:
    head = read_git(path, "rev-parse", "--verify", "HEAD")
    status = read_git(path, "status", "--porcelain=v1", "-z", "--untracked-files=all", "--ignore-submodules=none")
    states.append({
        "actual_head": head.strip() if head is not None else None,
        "worktree": ("unknown" if status is None else "dirty" if status else "clean"),
        "status_entries": status.rstrip("\0").split("\0") if status else [],
    })
print(json.dumps(states, ensure_ascii=True))
"""


def prepare_colab_release(execute, log_event, *, notebook, notebook_updated_at, experiment_id, transfer_id,
                          alignment, benchmark, logs, persistent):
    """Checkout only the pinned SHAs, record both observations, and fail closed."""
    checkouts = dict(alignment=alignment, benchmark_runtime=benchmark)
    errors = []
    for key, (path, url, revision) in checkouts.items():
        if not re.fullmatch(r"[0-9a-f]{40}", revision):
            errors.append(f"{key}: expected a full lowercase commit SHA")
    if not errors:
        for key, (path, url, revision) in checkouts.items():
            label = key.replace("_", "-")
            try:
                if not (path / ".git").exists():
                    execute("git", "clone", "--no-checkout", url, path, name="clone-" + label)
                execute("git", "-C", path, "fetch", "origin", revision, name="fetch-" + label)
                execute("git", "-C", path, "checkout", "--detach", revision, name="checkout-" + label)
            except Exception:
                errors.append(f"{key}: Git checkout failed; see the command log")
    try:
        result = execute(sys.executable, "-c", GIT_CHECKOUT_STATE,
                         *(value[0] for value in checkouts.values()),
                         name="inspect-colab-revisions", capture_json=True)
        states = json.loads(result.stdout)
    except Exception:
        errors.append("could not inspect the repositories; see the command log")
        states = [{"actual_head": None, "worktree": "unknown", "status_entries": []} for _ in checkouts]
    repositories = {}
    for (key, (_, _, pin)), state in zip(checkouts.items(), states, strict=True):
        repositories[key] = {"pinned_revision": pin, **state}
        if state["actual_head"] != pin:
            errors.append(f"{key}: HEAD differs from the pinned revision")
        if state["worktree"] != "clean":
            errors.append(f"{key}: worktree is {state['worktree']}")
    report = dict(schema="colab-release-v2", run_id=uuid.uuid4().hex,
                  started_utc=datetime.now(timezone.utc).isoformat(), notebook=notebook,
                  notebook_updated_at=notebook_updated_at, experiment_id=experiment_id,
                  transfer_experiment_id=transfer_id, status="FAIL" if errors else "PASS",
                  repositories=repositories, errors=errors)
    payload = json.dumps(report, ensure_ascii=True, indent=2, allow_nan=False) + "\n"
    filename = "colab-release-" + report["run_id"] + ".json"
    for root in (Path(logs), Path(persistent) / "execution-provenance"):
        root.mkdir(parents=True, exist_ok=True)
        with (root / filename).open("x", encoding="utf-8") as target:
            target.write(payload)
    log_event("colab-revision-check", report["status"], logs, json.dumps(report, ensure_ascii=True))
    print(payload, end="", flush=True)
    if errors:
        raise RuntimeError("Colab revision check failed. Resolve the reported Git state before running the experiment.")
    return report

# Start the revision check
RELEASE_RECORD = None
RELEASE_RECORD = prepare_colab_release(
    lambda *args, **kwargs: execute(*args, bootstrap=True, **kwargs), log_event,
    notebook='evaluate', notebook_updated_at=NOTEBOOK_UPDATED_AT,
    experiment_id=EXPERIMENT_ID, transfer_id=TRANSFER_EXPERIMENT_ID,
    alignment=(REPO, REPO_URL, REPO_REVISION),
    benchmark=(BENCHMARK_RUNTIME, BENCHMARK_RUNTIME_URL, BENCHMARK_RUNTIME_REVISION),
    logs=LOGS, persistent=PERSISTENT)


In [ ]:
#@title Set up Python and fairseq
os.chdir(REPO)
execute(sys.executable, '-m', 'pip', 'install', 'uv', name='install-uv')
execute('uv', 'venv', '--python', '3.10.22', '--seed', '--allow-existing', '/content/alignment-venv', name='create-python-environment')
for packages in [
    ['pip==23.3.2', 'setuptools<81', 'wheel'],
    ['torch==2.7.1', 'torchaudio==2.7.1', 'numpy==1.26.4', 'Cython==3.2.9'],
    ['--no-build-isolation', '-r', 'requirements-fairseq.txt'],
    ['-r', 'requirements.txt'],
]:
    execute(PYTHON, '-m', 'pip', 'install', *packages)


In [ ]:
#@title Install the pinned benchmark runtime
execute(PYTHON, '-m', 'pip', 'install', 'PyYAML==6.0.2', '--no-deps', '-e', str(BENCHMARK_RUNTIME))
execute(PYTHON, '-m', 'pip', 'check', name='dependency-check')


In [ ]:
#@title Load corpus settings

config = execute_json(PYTHON, '-c',
    "import json,sys,yaml; print(json.dumps(yaml.safe_load(open(sys.argv[1], encoding='utf-8'))))",
    DATA_CONFIG, name='data-settings')
needs_speech = globals().get('PREPARE_S2UT', globals().get('EVALUATE_S2UT', False))
root_value = CORPUS_ROOT_OVERRIDE or config.get('corpus_root')
manifest_value = SOURCE_MANIFEST_OVERRIDE or config.get('accepted_manifest')
summary_value = RELEASE_SUMMARY_OVERRIDE or config.get('release_summary')
if needs_speech and (not root_value or not manifest_value):
    raise ValueError('Set the corpus root and manifest in DATA_CONFIG, or set both CORPUS_ROOT_OVERRIDE and SOURCE_MANIFEST_OVERRIDE.')
CORPUS_ROOT = Path(root_value) if root_value else None
SOURCE_MANIFEST = Path(manifest_value) if manifest_value else None
RELEASE_SUMMARY = Path(summary_value) if summary_value else None
if CORPUS_ROOT and any(path.resolve().is_relative_to(CORPUS_ROOT.resolve()) for path in (Path(ROOT), Path(PERSISTENT))):
    raise ValueError('Choose an output directory outside the source corpus.')
os.environ.pop('S2ST_INPUT_MAP', None)
execute(PYTHON, '-m', 'pip', 'install', 'editdistance==0.8.1', 'librosa==0.10.2.post1',
    'accelerate==1.12.0', 'pandas==2.2.3', 'psutil==7.2.2', name='runtime-dependencies')
execute(PYTHON, '-m', 'pip', 'check', name='dependency-check')


In [ ]:
#@title Check runtime dependencies and CUDA

if SYNC_SAVED_TRANSFER_BLEU:
    log_event('evaluation-environment', 'skipped', LOGS, 'saved BLEU sync uses only the W&B SDK')
else:
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.environment', '--native', '--benchmark', '--cuda',
        name='evaluation-environment')

In [ ]:
#@title Run, save, and restore helpers
def corpus_stage(name, module, *arguments, work, restore_only=False):
    command = [PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run', '--work', str(work),
        '--backup', str(Path(PERSISTENT) / 'stages' / name)]
    if restore_only:
        command.append('--restore-only')
    execute(*command, '--', PYTHON, '-m', module, *map(str, arguments), name=name)

def corpus_common(restore_only=False):
    os.environ['CORPUS_ROOT'] = str(CORPUS_ROOT)
    summary_args = ['--summary', str(RELEASE_SUMMARY)] if RELEASE_SUMMARY else []
    corpus_stage('common', 'alignment_transfer_s2ut.benchmark_data', 'corpus',
        '--source', SOURCE_MANIFEST, '--corpus-root', CORPUS_ROOT, *summary_args,
        '--output', COMMON, '--resume', work=COMMON, restore_only=restore_only)
    lock = json.loads((COMMON / 'dataset-lock.json').read_text(encoding='utf-8'))
    expected = config.get('expected_pairs')
    if expected is not None and lock['total_pairs'] != expected:
        raise ValueError(f"train/dev count {lock['total_pairs']} differs from expected_pairs={expected}")
    print({'pairs': lock['total_pairs'], 'splits': lock['splits'],
        'estimated_waveform_GiB': round(lock['wav_bytes_estimate'] / 1024**3, 1)})

def corpus_stage_audio(create=False):
    staged = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'stage-corpus',
        '--common', str(COMMON), '--corpus-root', str(CORPUS_ROOT), '--packs', str(Path(PERSISTENT) / 'wav-packs'),
        '--local', str(Path(ROOT) / 'staged-inputs'), *(['--create-audio'] if create else ['--stream-audio']), name='audio-staging')
    if not create:
        os.environ['S2ST_INPUT_MAP'] = staged['input_map']
    print(staged)

def corpus_prepare(restore_only=False):
    corpus_stage('unit-extraction', 'alignment_transfer_s2ut.corpus_runtime', 'units',
        '--common', COMMON, '--output', BENCHMARK_UNITS, '--cache', '/content/alignment-assets',
        work=BENCHMARK_UNITS, restore_only=restore_only)
    corpus_stage('fairseq-data', 'alignment_transfer_s2ut.corpus_runtime', 'prepare',
        '--common', COMMON, '--units', BENCHMARK_UNITS, '--output', PREPARED,
        work=PREPARED, restore_only=restore_only)

def corpus_baseline(restore_only=False):
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.corpus_runtime', 'baseline',
        '--data', str(DATA), '--work', str(RESTORED), '--backup', str(BACKUP),
        *(['--restore-only'] if restore_only else ['--train']), name='baseline')
def run(name, module, *arguments, gpu=False, env=None, restore_only=False):
    arguments = list(map(str, arguments))
    payload = [PYTHON, '-m', module, *arguments]
    # Read-only environment checks and asset downloads have no stage output.
    if '--output' in arguments and module != 'alignment_transfer_s2ut.benchmark' or (
            module == 'alignment_transfer_s2ut.benchmark' and arguments[0] != 'plan'):
        if '--output' in arguments:
            output = Path(arguments[arguments.index('--output') + 1])
            work = output.parent if output.suffix in ('.json', '.jsonl', '.png') else output
            wrapper = [PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run',
                '--work', str(work), '--backup', str(Path(PERSISTENT) / 'stages' / name)]
            if restore_only:
                wrapper.append('--restore-only')
            if module == 'alignment_transfer_s2ut.train_t2tt' or module == 'alignment_transfer_s2ut.benchmark' and arguments[0] == 'train':
                wrapper.append('--training')
            payload = [*wrapper, '--', *payload]
    command = [PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'run', '--name', name, '--logs', str(LOGS)]
    if gpu:
        command.append('--monitor-gpu')
    return execute(*command, '--', *payload, name='notebook:' + name, env=env)

## Optional T2TT external BLEU

Set `EVALUATE_T2TT_EXTERNAL=True` to evaluate on the saved 8C set of 300 samples. For T2TT only, set `EVALUATE_S2UT=False`.

In [ ]:
#@title Evaluate T2TT on the external text set

if EVALUATE_T2TT_EXTERNAL:
    reference = REFERENCE_BENCHMARK or config.get('benchmark_reference_root')
    if not reference:
        raise ValueError('Set REFERENCE_BENCHMARK to the experiment root containing the saved external evaluation set.')
    text_work = Path('/content/alignment-transfer-work') / T2TT_RUN_ID / ('t2tt-' + T2TT_TOKENIZER)
    text_backup = Path('/content/drive/MyDrive/alignment-transfer-s2ut-data') / T2TT_RUN_ID / 'stages' / ('t2tt-' + T2TT_TOKENIZER)
    execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.evaluate_t2tt_external', 'restore-inputs',
        '--work', str(text_work), '--backup', str(text_backup), '--expected-updates', str(T2TT_MAX_UPDATES),
        name='restore-external-text-model')
    fixed_set = Path(ROOT) / 'external-text/fixed-set'
    source_args = ['--source-8c', SOURCE_8C] if SOURCE_8C else []
    run('external-text-fixed-set', 'alignment_transfer_s2ut.evaluate_t2tt_external', 'prepare',
        '--benchmark', reference, '--output', fixed_set, *source_args)
    output = Path(ROOT) / 'external-text' / T2TT_RUN_ID / T2TT_TOKENIZER / 'report/text-bleu.json'
    run('external-text-bleu-' + T2TT_RUN_ID + '-' + T2TT_TOKENIZER, 'alignment_transfer_s2ut.evaluate_t2tt_external', 'evaluate',
        '--fixed-set', fixed_set, '--checkpoint', text_work / 'checkpoint_best.pt',
        '--output', output, '--device', 'cuda', '--resume', gpu=True)
    result = json.loads(output.read_text(encoding='utf-8'))
    print(json.dumps(result, ensure_ascii=False, indent=2))
else:
    log_event('external-text-bleu', 'skipped', LOGS, 'EVALUATE_T2TT_EXTERNAL=False')

## 2. Restore inputs and Baseline

Restore completed preprocessing, then fetch and extract the audio ZIPs needed for evaluation.

In [ ]:
#@title Restore speech data and WAV ZIPs

if EVALUATE_S2UT:
    corpus_common(restore_only=True)
    corpus_stage_audio()
    corpus_prepare(restore_only=True)
    if not BACKFILL_TRANSFER_BLEU:
        corpus_baseline(restore_only=True)
    else:
        log_event('restore-baseline', 'skipped', LOGS, 'backfill only reads the Baseline receipt')
else:
    log_event('restore-inputs', 'skipped', LOGS, 'S2UT evaluation disabled')


### Backfill transfer BLEU from published checkpoints

Enable `BACKFILL_TRANSFER_BLEU` to score exact checkpoints and save them on Drive. The training reporter picks up saved results. Set `SYNC_SAVED_TRANSFER_BLEU=True` and select the transfer conditions to send saved BLEU without scoring again, after their training cells have stopped. The following sections are for full S2UT evaluation.


In [ ]:
#@title Backfill Aux-BLEU and ASR-BLEU or sync saved scores
def prepare_wandb():
    from google.colab import userdata
    try:
        key = userdata.get('WANDB_API_KEY')
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        key = None
    if not key or not WANDB_PROJECT.strip():
        log_event('wandb', 'skipped', LOGS, 'Set the WANDB_API_KEY Colab secret and a project name')
        return None
    directory = Path('/content/alignment-wandb-venv')
    python = directory / 'bin/python'
    import subprocess
    try:
        execute('uv', 'venv', '--python', PYTHON, '--allow-existing', directory, name='wandb-environment')
        execute('uv', 'pip', 'install', '--python', python, 'wandb==0.31.0', 'filelock==4.1.0', name='wandb-install')
    except (OSError, subprocess.CalledProcessError):
        log_event('wandb', 'failed', LOGS, 'SDK setup failed. See the setup log. Training metrics stay local.')
        return None
    return python, key

if BACKFILL_TRANSFER_BLEU or SYNC_SAVED_TRANSFER_BLEU:
    output = Path(PERSISTENT) / 'transfer-bleu' / (TRANSFER_EXPERIMENT_ID or 'default')
    conditions = [('distillation', COMPARE_DISTILLATION),
                  ('annealed_distillation', COMPARE_ANNEALED_DISTILLATION),
                  ('mask_distillation', COMPARE_MASK_DISTILLATION)]
    tracking = prepare_wandb() if SYNC_SAVED_TRANSFER_BLEU else None
    update = ['--update', TRANSFER_BLEU_UPDATE] if TRANSFER_BLEU_UPDATE else []
    for method, selected in conditions:
        if not selected:
            log_event('bleu-backfill-' + method, 'skipped', LOGS, 'condition is off')
            continue
        backup = Path(PERSISTENT) / 'stages' / ('research-v3-' + method + SCOPE_SUFFIX)
        if BACKFILL_TRANSFER_BLEU:
            execute(PYTHON, REPO / 'tools/transfer_bleu.py', 'backfill', '--method', method,
                    '--work', RESEARCH_ROOT / method, '--backup', backup, '--baseline-backup', BACKUP,
                    '--scratch', Path(ROOT) / 'transfer-bleu-work', '--output', output, '--logs', LOGS,
                    *update, name='bleu-backfill-' + method)
            log_event('wandb-bleu-' + method, 'queued', LOGS, 'Saved scores are available to the training reporter')
        elif tracking:
            python, key = tracking
            execute(python, REPO / 'tools/wandb_transfer.py', '--sync-bleu-only', '--method', method,
                    '--work', RESEARCH_ROOT / method, '--backup', backup, '--bleu-results', output, '--logs', LOGS,
                    '--project', WANDB_PROJECT.strip(), '--entity', WANDB_ENTITY.strip(),
                    env=dict(os.environ, WANDB_API_KEY=key), name='wandb-sync-saved-bleu-' + method)
else:
    log_event('bleu-backfill', 'skipped', LOGS, 'backfill is off')


In [ ]:
#@title Restore and register the Baseline checkpoint

if EVALUATE_S2UT and not BACKFILL_TRANSFER_BLEU:
    run('register-baseline', 'alignment_transfer_s2ut.benchmark', 'register',
        '--checkpoint', RESTORED / 'checkpoints/checkpoint_best.pt', '--data', PREPARED,
        '--recipe-lock', RESTORED / 's2ut-recipe-lock.json', '--metadata', RESTORED / 'research-metadata.json',
        '--output', BASELINE, '--resume')
    baseline = json.loads(BASELINE.read_text(encoding='utf-8'))
    print({key:baseline[key] for key in ('checkpoint_updates','training_budget','selection_policy')})
    BACKBONE_CONFIG.write_text(json.dumps({'backbone':baseline['backbone']}),encoding='utf-8')
else:
    log_event('registration', 'skipped', LOGS, 'S2UT full evaluation disabled or backfill selected')


In [ ]:
#@title Restore the speech manifest and unit artifacts

if EVALUATE_S2UT and not BACKFILL_TRANSFER_BLEU:
    run('full-manifest', 'alignment_transfer_s2ut.benchmark_data', 'manifest',
        '--common', COMMON, '--prepared', PREPARED, '--output', MANIFEST, '--resume', restore_only=True)
    run('import-units', 'alignment_transfer_s2ut.benchmark_data', 'units', '--manifest', MANIFEST,
        '--benchmark-units', BENCHMARK_UNITS, '--prepared', PREPARED,
        '--output', Path(ROOT) / 'units', '--resume', restore_only=True)
else:
    log_event('restore-speech-artifacts', 'skipped', LOGS, 'S2UT full evaluation disabled or backfill selected')


## 3. Restore models for comparison

In [ ]:
#@title Restore the selected transfer checkpoints

if EVALUATE_S2UT and not BACKFILL_TRANSFER_BLEU:
    RESEARCH_CONDITIONS = [(method,label,globals()['COMPARE_' + method.upper()]) for method,label in METHOD_LABELS]
    for method,label,enabled in RESEARCH_CONDITIONS:
        if enabled:
            execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'restore-training',
                '--work', str(RESEARCH_ROOT / method),
                '--backup', str(Path(PERSISTENT) / 'stages' / ('research-v3-' + method + SCOPE_SUFFIX)),
                '--method', method, name='restore-' + method)
        else:
            log_event(method, 'skipped', LOGS, 'condition excluded from evaluation')
else:
    log_event('restore-models', 'skipped', LOGS, 'S2UT full evaluation disabled or backfill selected')


## 4. Shared inference and dev evaluation

Lower UER and higher ASR-BLEU are better. Enable waveform generation and ASR-BLEU in the settings.

In [ ]:
#@title Generate translations and score the dev split

if EVALUATE_S2UT and not BACKFILL_TRANSFER_BLEU:
    conditions = []
    from IPython.display import Markdown, display

    if not GENERATE_WAVEFORMS:
        log_event('waveform-generation', 'skipped', LOGS, 'GENERATE_WAVEFORMS=False')
    if not ASR_BLEU:
        log_event('asr-bleu', 'skipped', LOGS, 'ASR_BLEU=False')

    transfer_conditions = []
    for method, label, enabled in RESEARCH_CONDITIONS:
        folder = RESEARCH_ROOT / method
        if enabled:
            transfer_conditions.append((method, label, folder, RESEARCH_ROOT / 'registrations' / (method + '.json'), 'research-v3-register-' + method + TRANSFER_SUFFIX))
    missing = [label for _, label, folder, _, _ in transfer_conditions
               if not (folder / 'checkpoints/checkpoint_best.pt').is_file()]
    if missing:
        log_event('dev-comparison', 'skipped', LOGS, 'Waiting for training to complete: ' + ', '.join(missing))
    else:
        conditions = [('baseline', 'Baseline', BASELINE)]
        for name, label, folder, record, registration in transfer_conditions:
            run(registration, 'alignment_transfer_s2ut.benchmark', 'register-transfer', '--run', folder,
                '--checkpoint', folder / 'checkpoints/checkpoint_best.pt', '--output', record, '--resume')
            conditions.append((name, label, record))
        records = [record for _, _, record in conditions]
        comparison_name = '-'.join(name for name, _, _ in conditions) + TRANSFER_SUFFIX
        COMPARISON_DIR = Path(ROOT) / 'comparison' / comparison_name
        if len(records) > 1:
            run(comparison_name + '-compare-training', 'alignment_transfer_s2ut.benchmark', 'compare', *records,
                '--output', COMPARISON_DIR / 'training-comparison.json', '--resume')
        source_only = Path(ROOT) / 'dev-source/dev-source.jsonl'
        run('dev-source', 'alignment_transfer_s2ut.prepare_inference_input', '--manifest', MANIFEST, '--split', 'dev', '--output', source_only, '--resume')
        if GENERATE_WAVEFORMS:
            run('vocoder', 'alignment_transfer_s2ut.fetch_assets', '--config', 'configs/s2ut/vocoder.yaml')
        evaluations = []
        table = ['| Condition | ASR-BLEU | UER |', '|---|---:|---:|']
        for name, label, record in conditions:
            predictions = Path(ROOT) / ('evaluation' + TRANSFER_SUFFIX) / name / 'dev-predictions.jsonl'
            vocoder = ['--vocoder-config', 'configs/s2ut/vocoder.yaml'] if GENERATE_WAVEFORMS else ['--units-only']
            run(name + '-infer' + TRANSFER_SUFFIX, 'alignment_transfer_s2ut.benchmark', 'infer', '--record', record, '--input', source_only,
                '--output', predictions, '--device', 'cuda', '--resume', *vocoder, gpu=True)
            metrics = predictions.with_name('dev-metrics.json')
            asr = ['--asr-config', 'configs/experiments/evaluation.yaml', '--device', 'cuda'] if ASR_BLEU else []
            run(name + '-evaluate' + TRANSFER_SUFFIX, 'alignment_transfer_s2ut.evaluate', '--manifest', MANIFEST, '--split', 'dev',
                '--units', Path(ROOT) / 'units', '--predictions', predictions, '--output', metrics, '--resume', *asr, gpu=ASR_BLEU)
            evaluations.append(metrics)
            values = json.loads(metrics.read_text())
            bleu = f"{values['asr_bleu']:.2f}" if values.get('asr_bleu') is not None else '—'
            uer = f"{values['unit_error_rate']:.4f}" if values.get('unit_error_rate') is not None else '—'
            table.append(f'| {label} | {bleu} | {uer} |')
        if len(records) > 1:
            run(comparison_name + '-comparison', 'alignment_transfer_s2ut.benchmark', 'compare', *records, '--evaluations', *evaluations,
                '--output', COMPARISON_DIR / 'comparison.json', '--resume')
            print('Comparison report:', COMPARISON_DIR / 'comparison.json')
        display(Markdown('\n'.join(table)))
else:
    log_event('evaluate', 'skipped', LOGS, 'S2UT full evaluation disabled or backfill selected')
